# Closed-world self-consistency check for best-member matching

**The question.** The retrieval run confirms 0.297 of what it proposed. Is that the method, or the
pool it was pointed at?

**Why the question is open.** The pool is residual by construction: crops the clustering refused to
assign *and* that manual curation had already passed over. 2,610 of them are crops a reviewer
looked at once and removed. A method could be excellent and still score poorly there.

**The evidence.** Members are withheld from the final, expanded catalogue, and the
method is asked to put them back. Ground truth is known for every withheld crop, so no review is
needed and no catalogue file is touched: the catalogue is read, never written.

**What this cannot establish.** The withheld crops are in their classes *because* a similarity
method in this same embedding space put them there. So the test population is selected by the
criterion under test, and the recall figures below are optimistic by construction: an impression
that resembles nothing in its class could never have joined it, and so can never appear here. What
remains informative is internal assignment precision, which asks a different question but is not
an external estimate.

---

**Reads** the verified catalogue and the frozen DINOv2 features · **Writes**
`3_retrieval_outputs/holdout_v1/` · **Chapter README** §5

## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
NOTEBOOK_DIR = Path.cwd().resolve()
assert NOTEBOOK_DIR.name == '3_retrieval', 'Run this notebook from its own folder, 3_retrieval/.'
PROJECT_DIR = NOTEBOOK_DIR.parent

FEATURE_RUN = 'all_regions_v1_minside24_dinov2_vitb14_binarized_v1'
FEATURE_DIR = PROJECT_DIR / 'feature_extraction_outputs' / FEATURE_RUN
CATALOGUE_DIR = PROJECT_DIR / 'Fleurons' / 'Fleurons_v2_plus_retrieval'
RETRIEVAL_DIR = PROJECT_DIR / '3_retrieval_outputs'

RUN_TAG = ''
ALLOW_OVERWRITE = False


def guarded_mkdir(path):
    """Create an output directory, refusing to overwrite a populated one."""
    if path.exists() and any(path.iterdir()) and not (RUN_TAG or ALLOW_OVERWRITE):
        raise RuntimeError(
            f'{path} already contains results. Set RUN_TAG to write elsewhere, '
            f'or ALLOW_OVERWRITE = True to replace them.')
    path.mkdir(parents=True, exist_ok=True)
    return path

OUTPUT_DIR = guarded_mkdir(RETRIEVAL_DIR / f'holdout_v1{RUN_TAG}')
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260806
HOLD_FRACTION = 0.30
REPEATS = 20
MIN_CLASS_SIZE = 5          # a 30% hold-out must still leave a usable reference set
THRESHOLDS = [0.80, 0.85, 0.90, 0.92, 0.95, 0.965]

manifest = pd.read_csv(FEATURE_DIR / 'features_manifest.csv')
features = np.load(FEATURE_DIR / 'dino_features_binarized.npy')
assert len(manifest) == len(features)
assert np.array_equal(manifest.feature_index.to_numpy(), np.arange(len(manifest)))
assert not manifest.crop_path.duplicated().any()
norms = np.linalg.norm(features, axis=1, keepdims=True)
assert (norms > 0).all(), 'Zero-norm descriptors cannot be cosine-normalised.'
features = features / norms
row_of = dict(zip(manifest.crop_path, manifest.feature_index))

catalogue = {}
for folder in sorted(p for p in CATALOGUE_DIR.iterdir() if p.is_dir()):
    rows = sorted({row_of[str(p.resolve().relative_to(PROJECT_DIR))] for p in folder.iterdir()
                   if str(p.resolve().relative_to(PROJECT_DIR)) in row_of})
    if len(rows) >= MIN_CLASS_SIZE:
        catalogue[folder.name.split(' (')[0]] = np.array(rows)

print(f'catalogue classes with at least {MIN_CLASS_SIZE} members in this feature run: {len(catalogue)}')
print(f'crops covered: {sum(len(v) for v in catalogue.values()):,}')
print(f'writing to: {OUTPUT_DIR.relative_to(PROJECT_DIR)}')

catalogue classes with at least 5 members in this feature run: 75
crops covered: 7,798
writing to: 3_retrieval_outputs/holdout_v1


## 2. Design

Each repeat withholds 30% of every eligible class at random as the test queries. The remaining 70% is the reference exemplar set they are scored against.
Each withheld crop is then scored by **best-member matching**: its maximum cosine against any
remaining member of any class, and assigned the class of that best match, exactly as the retrieval
run does. Two quantities follow directly, and neither needs a reviewer, since the true class of
every withheld crop is known:

- **precision**: of the withheld crops retrieved at or above a threshold, the share assigned to
  their own class;
- **recall**: of all withheld crops, the share both retrieved and correctly assigned.

The experiment is repeated 20 times with different random splits, so the figures carry a spread
rather than resting on one draw. Every class with at least five members is used rather than a
sample of classes, which removes the question of how a class sample was chosen. The split unit is
a crop, not a scan or work, so same-scan and same-work siblings can remain among the references.
The result is therefore a catalogue self-consistency check rather than evidence of generalisation
to a new scan or work.

In [2]:
rng = np.random.default_rng(RANDOM_SEED)
records, confusions = [], []

for repeat in range(REPEATS):
    reference_rows, held_rows, held_truth = {}, [], []
    for name, rows in catalogue.items():
        shuffled = rng.permutation(rows)
        cut = max(1, int(round(HOLD_FRACTION * len(rows))))
        held_rows.extend(shuffled[:cut].tolist())
        held_truth.extend([name] * cut)
        reference_rows[name] = shuffled[cut:]

    held_rows = np.asarray(held_rows)
    held_truth = np.asarray(held_truth)
    held_features = features[held_rows]

    best_score = np.full(len(held_rows), -1.0)
    best_class = np.empty(len(held_rows), dtype=object)
    for name, reference in reference_rows.items():
        sims = (held_features @ features[reference].T).max(axis=1)
        improved = sims > best_score
        best_score[improved] = sims[improved]
        best_class[improved] = name

    correct = best_class == held_truth
    for truth, predicted in zip(held_truth[~correct], best_class[~correct]):
        confusions.append(tuple(sorted((truth, predicted))))

    for threshold in THRESHOLDS:
        retrieved = best_score >= threshold
        records.append({
            'repeat': repeat, 'threshold': threshold,
            'held_out': len(held_rows), 'retrieved': int(retrieved.sum()),
            'correct': int((retrieved & correct).sum()),
            'precision': float((retrieved & correct).sum() / retrieved.sum()),
            'recall': float((retrieved & correct).sum() / len(held_rows)),
        })

repeats_frame = pd.DataFrame(records)
repeats_frame.to_csv(OUTPUT_DIR / 'holdout_repeats.csv', index=False)

summary = (repeats_frame.groupby('threshold')
           .agg(withheld=('held_out', 'mean'),
                retrieved=('retrieved', 'mean'),
                precision=('precision', 'mean'),
                precision_low=('precision', 'min'),
                recall=('recall', 'mean'),
                recall_low=('recall', 'min'),
                recall_high=('recall', 'max'))
           .round(4))
summary.to_csv(OUTPUT_DIR / 'holdout_summary.csv')
summary

,withheld,retrieved,precision,precision_low,recall,recall_low,recall_high
threshold,,,,,,,
0.800,2342.0,2341.65,0.9968,0.9953,0.9966,0.9949,0.9983
0.850,2342.0,2338.25,0.9969,0.9953,0.9953,0.9927,0.9974
0.900,2342.0,2308.40,0.9971,0.9957,0.9828,0.9787,0.9863
0.920,2342.0,2230.95,0.9975,0.9955,0.9502,0.9381,0.9607
0.950,2342.0,1683.85,0.9999,0.9994,0.7189,0.6994,0.7297
0.965,2342.0,1039.30,1.0000,1.0000,0.4438,0.4330,0.4586


**Takeaway.** Precision sits at **0.997** and does not move materially across the operating range:
when best-member matching retrieves a crop at all, it almost always assigns it to the right class.
Recall is what the threshold buys or spends, 0.997 at cosine 0.80, 0.983 at 0.90, 0.950 at 0.92,
and then a sharp fall to 0.719 at 0.95. The spread across 20 splits is narrow, so the result is
not peculiar to one crop-level partition of this catalogue.

**The two figures carry different weight, and the difference is not the one it appears to be.**

*Precision is more informative than recall here, but remains internal.* It asks whether a
withheld crop is closer to a member of its own class than to a member of any of the other 74. The
competing classes were built the same way as its own, so nothing about the construction guarantees
the answer, and the 152 misassignments of Section 3 show it is not guaranteed in practice. Yet
catalogue selection and crop-level splitting still make the result optimistic.

*Recall does not survive it.* Only crops that already resemble their classmates can be in a class,
so only such crops can be withheld. The figures above describe recovery of easy members, not of the
impressions the residual pool actually holds.

The honest reading is therefore two conditioned results, not a statistical bracket: **0.297** is
the confirmation rate of the selected residual-pool shortlist, while **0.997** is optimistic
closed-world assignment precision on catalogue members. Their difference is consistent with a
difficult residual pool but does not isolate that cause.

## 3. Where the errors fall

152 misassignments occur across the 20 repeats, out of roughly 2,340 withheld crops per repeat. They
are not spread evenly over the catalogue.

In [3]:
from collections import Counter

counted = Counter(confusions)
confusion_frame = (pd.DataFrame([{'class A': a, 'class B': b, 'errors': n}
                                 for (a, b), n in counted.items()])
                   .sort_values('errors', ascending=False)
                   .reset_index(drop=True))
confusion_frame.to_csv(OUTPUT_DIR / 'holdout_confusions.csv', index=False)

total_errors = sum(counted.values())
top = confusion_frame.head(10)
print(f'misassignments over {REPEATS} repeats : {total_errors}')
print(f'distinct design pairs involved     : {len(counted)}')
print(f'share from the ten worst pairs     : {top.errors.sum() / total_errors:.1%}')
top

misassignments over 20 repeats : 152
distinct design pairs involved     : 34
share from the ten worst pairs     : 69.1%


,class A,class B,errors
0,Fleuron_17,Fleuron_35,24
1,Fleuron_1,Fleuron_4,18
2,Fleuron_4,Fleuron_79,13
3,Fleuron_1,Fleuron_3,10
4,Fleuron_16,Fleuron_48,10
5,Fleuron_48,Fleuron_49,7
6,Fleuron_3,Fleuron_4,6
7,Fleuron_24,Fleuron_66,6
8,Fleuron_50,Fleuron_53,6
9,Fleuron_11,Fleuron_69,5


**Takeaway.** Thirty-four pairs of classes account for every error, and the ten worst account for
69% of them. Within this eligible catalogue subset and crop-level protocol, observed errors are
concentrated in a small, enumerable set of designs.

The clustering chapter reaches the same limitation by a different route. Its
`fig:clu-confusable` shows crop pairs whose embeddings sit above the identity threshold and which
a reviewer judged to be different designs, differing in nothing but a lobe count or an interior
element. Two methods, evaluated by two different procedures on two different samples, fail on the
same *kind* of design, which locates the limitation in the shared DINOv2 representation rather than
in either algorithm: silhouette survives the embedding, interior detail and countable structure do
not.

The practical consequence is narrow and worth stating. Any historical claim resting on one of these
design pairs needs the two classes checked against each other by eye before the count is published.
Classes outside the list were not flagged by this test; they are not certified confusion-free,
especially the 17 classes excluded by the minimum-size rule.

## 4. What this notebook establishes

**Best-member matching is internally consistent on ordinary catalogue members**: 0.997 precision
across 20 random crop-level splits of all 75 eligible classes. The figure is optimistic rather than
external validation, because the same representation helped assemble the catalogue and siblings can
cross the split. It is still informative: the test discriminates among 75 competing classes and
records 152 failures.

**Its recall figures are optimistic by construction** and are reported for the shape of the
threshold trade rather than as a prediction of what retrieval recovers in the corpus.

**The low confirmation rate of the main run is consistent with pool difficulty**, but this check
does not establish that cause. The chapter reports 0.297 and 0.997 as answers to different
conditioned questions, not as competing estimates of one quantity.

**Observed errors are concentrated in 34 pairs among the 75 eligible classes**, a failure pattern
the clustering chapter also observes in the shared representation. A scan- or work-disjoint test,
macro-per-class metrics and an exemplar-count control remain outstanding.